# Stage 2+ — Upgraded SFT (built to beat Crane decisively)

Same proven machinery as your working Stage 2, with the quality levers stacked
on top to clear **27.6 BLEU / 56.8 chrF++** by a margin — while staying at **1B**
so it's a win on the efficiency ratio, their own game.

### What's new vs your last run
| Lever | Change |
|---|---|
| **MAFAND-MT** (human news, en-swa) | +34k gold pairs, news-domain like FLORES |
| **Full SALT** | uncapped (KenCorpus-curated) instead of 20k |
| **Back-translation** | ~15k synthetic English→*real* Swahili via NLLB on your mono corpus |
| **Quality filter** | length-ratio filter drops misaligned pairs |
| **Fixed eval** | 256 new tokens + beam 8 + length-penalty (no truncation deflation) |
| **3 epochs** | up from 2 |

Everything else is unchanged and proven: fp16, SDPA, packing, checkpoint-resume
to the Hub, torchao removed, no `-U`.

> **Setup:** GPU **T4 ×2** (not P100), Internet ON, `HF_TOKEN` secret set, FLORES+
> terms accepted. Points at your `swa-gemma-cpt-merged`. Resume across sessions
> as before — this run is bigger, so expect several 12h windows.

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
%%capture
# Only add what's missing; never -U (that breaks torch). Remove old torchao (breaks merge).
!pip install -q sacrebleu bitsandbytes
!pip uninstall -y torchao

In [ ]:
import torch, transformers, peft
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
print("HF auth OK | CUDA:", torch.cuda.is_available())

## Configuration

In [ ]:
HF_USERNAME    = "Dev-Noble"                             # <-- your HF username
CPT_MERGED     = f"{HF_USERNAME}/swa-gemma-cpt-merged"    # output of Stage 1
HUB_SFT_CKPT   = f"{HF_USERNAME}/swa-gemma-sft2"          # rolling checkpoints (new name)
HUB_SFT_MERGED = f"{HF_USERNAME}/swa-gemma-sft2-merged"   # final model

MAX_LEN        = 512
LR, LR_SCHED   = 2e-4, "cosine"
NUM_EPOCHS     = 3
PER_DEV_BS     = 2
GRAD_ACCUM     = 16        # effective batch 32
WARMUP_RATIO   = 0.03
SAVE_STEPS     = 200
LOG_STEPS      = 20
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05
LORA_TARGETS = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

# Parallel-data caps (None = all). MAFAND + SALT are gold -> take all.
CAP = {"mafand": None, "salt": None, "rogendo": 60_000,
       "inkuba_mmt": 30_000, "alpaca": 15_000, "aya": None}

# Back-translation
BT_ENABLE = True
BT_ROWS   = 15_000        # synthetic English -> real Swahili pairs

OUTPUT_DIR = "/kaggle/working/sft2_out"
os.makedirs(OUTPUT_DIR, exist_ok=True)
assert HF_USERNAME != "YOUR_HF_USERNAME", "Set HF_USERNAME."
print("effective batch:", PER_DEV_BS*GRAD_ACCUM, "| epochs:", NUM_EPOCHS)

## Tokenizer + prompt format

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(CPT_MERGED)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def translation_prompt(english, swahili):
    return ("<start_of_turn>user\n"
            "Translate the following English sentence to Swahili.\n"
            f"English: {str(english).strip()}\nSwahili:<end_of_turn>\n"
            f"<start_of_turn>model\n{str(swahili).strip()}<end_of_turn>")

def instruction_prompt(instruction, response, input_text=""):
    user = str(instruction).strip() + (f"\n{str(input_text).strip()}" if str(input_text).strip() else "")
    return (f"<start_of_turn>user\n{user}<end_of_turn>\n"
            f"<start_of_turn>model\n{str(response).strip()}<end_of_turn>")

def translation_prompt_infer(english):
    return ("<start_of_turn>user\n"
            "Translate the following English sentence to Swahili.\n"
            f"English: {str(english).strip()}\nSwahili:<end_of_turn>\n<start_of_turn>model\n")
print("ok")

## Load parallel + instruction data (now with MAFAND)

In [ ]:
from datasets import load_dataset
prompts = []
def sp(name): 
    n = CAP.get(name); return f"train[:{n}]" if n else "train"

# --- MAFAND-MT en-swa: human news. HF script loader is dead (datasets dropped
#     script support), so load the JSON data files directly from GitHub. ---
try:
    base = "https://raw.githubusercontent.com/masakhane-io/lafand-mt/main/data/json_files/en-swa"
    d = load_dataset("json",
                     data_files={"tr": f"{base}/train.json", "dv": f"{base}/dev.json"},
                     split="tr+dv")
    b = len(prompts)
    for r in d:
        t = r["translation"]; en, sw = t.get("en"), t.get("swa")
        if en and sw and len(str(en)) > 5 and len(str(sw)) > 5:
            prompts.append(translation_prompt(en, sw))
    print("MAFAND:", len(prompts) - b)
except Exception as e:
    print("MAFAND FAILED:", e)

# --- Rogendo EN-SW pairs ---
try:
    d = load_dataset("Rogendo/English-Swahili-Sentence-Pairs", split=sp("rogendo"))
    b=len(prompts)
    for r in d:
        en, sw = r.get("English sentence",""), r.get("Swahili Translation","")
        if len(str(en))>8 and len(str(sw))>8: prompts.append(translation_prompt(en, sw))
    print("Rogendo:", len(prompts)-b)
except Exception as e: print("Rogendo FAILED:", e)

# --- Sunbird SALT (full) ---
try:
    d = load_dataset("Sunbird/salt", "text-all", split=sp("salt"))
    b=len(prompts)
    for r in d:
        en, sw = r.get("eng_source_text",""), r.get("swa_text","")
        if len(str(en))>5 and len(str(sw))>5: prompts.append(translation_prompt(en, sw))
    print("SALT:", len(prompts)-b)
except Exception as e: print("SALT FAILED:", e)

# --- Inkuba-Swahili-MMT (SW->EN; swap) ---
try:
    d = load_dataset("Svngoku/Inkuba-Swahili-MMT", split=sp("inkuba_mmt"))
    b=len(prompts)
    for r in d:
        en, sw = r.get("targets",""), r.get("inputs","")
        if len(str(en))>8 and len(str(sw))>8: prompts.append(translation_prompt(en, sw))
    print("Inkuba-MMT:", len(prompts)-b)
except Exception as e: print("Inkuba-MMT FAILED:", e)

# --- Aya (human Swahili instructions) ---
try:
    d = load_dataset("CohereForAI/aya_dataset", split="train")
    d = d.filter(lambda x: str(x.get("language","")).lower() in ["swahili","sw"])
    b=len(prompts)
    for r in d:
        if len(str(r.get("targets","")))>8:
            prompts.append(instruction_prompt(r.get("inputs",""), r.get("targets","")))
    print("Aya:", len(prompts)-b)
except Exception as e: print("Aya FAILED:", e)

# --- Alpaca-Swahili (structured tasks) ---
try:
    d = load_dataset("iamshnoo/alpaca-cleaned-swahili", split=sp("alpaca"))
    b=len(prompts)
    for r in d:
        if r.get("instruction") and len(str(r.get("output","")))>8:
            prompts.append(instruction_prompt(r.get("instruction",""), r.get("output",""), r.get("input","")))
    print("Alpaca:", len(prompts)-b)
except Exception as e: print("Alpaca FAILED:", e)

print("Parallel+instruction subtotal:", len(prompts))

## Back-translation — manufacture fluent-Swahili-target pairs with NLLB

Translate monolingual Swahili → English, then use each (synthetic-English → real-Swahili)
pair for training. The Swahili side is genuine, which is what lifts output quality.

In [ ]:
if BT_ENABLE:
    import gc
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer as AT
    print("Loading NLLB for back-translation...")
    mono = load_dataset("AlexLeoTz/jamiiforums-2b", split=f"train[:{BT_ROWS*3}]")
    sw_lines = [r["text"].strip() for r in mono
                if 40 <= len(str(r.get("text","")).strip()) <= 280][:BT_ROWS]
    print("mono Swahili lines:", len(sw_lines))

    ntok = AT.from_pretrained("facebook/nllb-200-distilled-600M")
    nllb = AutoModelForSeq2SeqLM.from_pretrained(
        "facebook/nllb-200-distilled-600M", torch_dtype=torch.float16).to("cuda").eval()
    eng_id = ntok.convert_tokens_to_ids("eng_Latn")
    ntok.src_lang = "swh_Latn"

    bt = []; B = 48
    for i in range(0, len(sw_lines), B):
        chunk = sw_lines[i:i+B]
        enc = ntok(chunk, return_tensors="pt", padding=True, truncation=True,
                   max_length=128).to("cuda")
        with torch.no_grad():
            g = nllb.generate(**enc, forced_bos_token_id=eng_id, max_new_tokens=128, num_beams=1)
        for en, sw in zip(ntok.batch_decode(g, skip_special_tokens=True), chunk):
            if len(en) > 8: bt.append(translation_prompt(en, sw))
        if i % (B*30) == 0: print(f"  BT {i}/{len(sw_lines)}")

    prompts += bt
    del nllb, ntok; gc.collect(); torch.cuda.empty_cache()
    print("Back-translated pairs added:", len(bt))
print("TOTAL before filter:", len(prompts))

## Quality filter (drop misaligned pairs) + shuffle

In [ ]:
import random, re
def ok_pair(p):
    # crude length-ratio sanity check on the English/Swahili spans
    m = re.search(r"English: (.*?)\nSwahili:", p, re.DOTALL)
    n = re.search(r"<start_of_turn>model\n(.*?)<end_of_turn>", p, re.DOTALL)
    if not (m and n): return True   # instruction-format rows: keep
    en, sw = m.group(1).strip(), n.group(1).strip()
    if len(en) < 3 or len(sw) < 3: return False
    r = len(sw) / max(len(en), 1)
    return 0.4 <= r <= 2.5          # keep plausible ratios

before = len(prompts)
prompts = [p for p in prompts if ok_pair(p)]
random.seed(42); random.shuffle(prompts)
print(f"filtered {before-len(prompts)} pairs | FINAL: {len(prompts)}")

## Tokenize and pack

In [ ]:
from datasets import Dataset
from itertools import chain

ds = Dataset.from_dict({"text": [p + tok.eos_token for p in prompts]})
tokd = ds.map(lambda b: tok(b["text"], add_special_tokens=True),
              batched=True, remove_columns=["text"], desc="tokenize")
def group(ex):
    ids = list(chain(*ex["input_ids"]))
    total = (len(ids)//MAX_LEN)*MAX_LEN; ids = ids[:total]
    blk = [ids[i:i+MAX_LEN] for i in range(0, total, MAX_LEN)]
    return {"input_ids": blk, "attention_mask": [[1]*MAX_LEN for _ in blk],
            "labels": [b[:] for b in blk]}
packed = tokd.map(group, batched=True, desc="pack")
print("packed blocks:", len(packed), "| ~tokens:", len(packed)*MAX_LEN)

## Load CPT-merged model (4-bit) + fresh SFT LoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                         bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(
    CPT_MERGED, quantization_config=bnb, device_map={"": 0},
    torch_dtype=torch.float16, attn_implementation="sdpa")
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGETS, bias="none"))
model.print_trainable_parameters()

## Resume-from-Hub + train

In [ ]:
from huggingface_hub import create_repo, list_repo_files, snapshot_download
from transformers import TrainingArguments, Trainer, default_data_collator

def fetch_resume(repo_id, local="/kaggle/working/resume_sft2"):
    try: files = list_repo_files(repo_id, repo_type="model")
    except Exception: print("No repo yet -> fresh."); return None
    if any(f.startswith("last-checkpoint/") for f in files):
        snapshot_download(repo_id, allow_patterns="last-checkpoint/*",
                          local_dir=local, repo_type="model")
        print("Resuming."); return os.path.join(local, "last-checkpoint")
    print("No last-checkpoint -> fresh."); return None

create_repo(HUB_SFT_CKPT, private=True, exist_ok=True)
resume_ckpt = fetch_resume(HUB_SFT_CKPT)

args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEV_BS, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, lr_scheduler_type=LR_SCHED, warmup_ratio=WARMUP_RATIO,
    fp16=True, bf16=False, optim="paged_adamw_8bit",
    logging_steps=LOG_STEPS, save_strategy="steps", save_steps=SAVE_STEPS,
    save_total_limit=2, report_to="none", dataloader_num_workers=2,
    dataloader_pin_memory=False, gradient_checkpointing=False,
    push_to_hub=True, hub_model_id=HUB_SFT_CKPT, hub_strategy="checkpoint",
    hub_private_repo=True)
trainer = Trainer(model=model, args=args, train_dataset=packed,
                  data_collator=default_data_collator)
trainer.train(resume_from_checkpoint=resume_ckpt)

## Merge final model

In [ ]:
import gc
from peft import PeftModel
adapter_dir = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.model.save_pretrained(adapter_dir); tok.save_pretrained(adapter_dir)
del model, trainer; gc.collect(); torch.cuda.empty_cache()
base = AutoModelForCausalLM.from_pretrained(CPT_MERGED, torch_dtype=torch.float16, device_map={"": 0})
merged = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
merged.push_to_hub(HUB_SFT_MERGED, private=True); tok.push_to_hub(HUB_SFT_MERGED)
print("Final model pushed to:", HUB_SFT_MERGED)

## FLORES-200 scorecard — fixed generation (256 tokens, beam 8)

In [ ]:
from datasets import load_dataset
def load_flores():
    try:
        en = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest")
        sw = load_dataset("openlanguagedata/flores_plus", "swh_Latn", split="devtest")
        return [r["text"] for r in en], [r["text"] for r in sw]
    except Exception as e:
        print("flores_plus failed (", e, ") -> mirror")
        en = load_dataset("Muennighoff/flores200", "eng_Latn", split="devtest")
        sw = load_dataset("Muennighoff/flores200", "swh_Latn", split="devtest")
        k = "sentence" if "sentence" in en.column_names else en.column_names[0]
        return [r[k] for r in en], [r[k] for r in sw]
src, ref = load_flores(); print("eval sentences:", len(src))

In [ ]:
import torch
merged.eval(); merged.config.use_cache = True
tok.padding_side = "left"

def batch_translate(sentences, bs=16, max_new=256, length_penalty=1.1):
    out=[]
    for i in range(0, len(sentences), bs):
        chunk = sentences[i:i+bs]
        enc = tok([translation_prompt_infer(s) for s in chunk], return_tensors="pt",
                  padding=True, truncation=True, max_length=MAX_LEN).to(merged.device)
        with torch.no_grad():
            g = merged.generate(**enc, max_new_tokens=max_new, num_beams=8,
                                do_sample=False, length_penalty=length_penalty,
                                early_stopping=True)
        out += [t.strip() for t in tok.batch_decode(g[:, enc["input_ids"].shape[1]:],
                                                     skip_special_tokens=True)]
        if i % (bs*10) == 0: print(f"  {i}/{len(sentences)}")
    return out
hyp = batch_translate(src)

In [ ]:
import sacrebleu
chrf = sacrebleu.corpus_chrf(hyp, [ref], word_order=2)
bleu = sacrebleu.corpus_bleu(hyp, [ref])
print("="*46)
print(f"  chrF++ : {chrf.score:5.2f}   (Crane: 56.8)")
print(f"  BLEU   : {bleu.score:5.2f}   (Crane: 27.6)")
print("="*46)
print("BEAT chrF++:", chrf.score > 56.8, "| BEAT BLEU:", bleu.score > 27.6)

## If you cleared it — you've beaten Crane at 1B. If you want more headroom:
**Stage 3 (DPO vs COMET-QE)** is the next lever — sample candidates from this model,
score with quality estimation, train it to prefer the best. That targets the residual
lexical/agreement errors and pushes toward the GPT-5-Mini tier (~32 / 62).